In [ ]:
import os
import sys

notebook_dir = os.path.abspath("")
print(f"Notebook directory: {notebook_dir}")

project_dir = os.path.dirname(notebook_dir)
print(f"Project directory: {project_dir}")

if project_dir not in sys.path:
    print("Project directory is not in system path!!\nAppending....")
    sys.path.append(project_dir)
    print("Done")

In [ ]:
import pandas as pd
import numpy as np

from src.config import get_config_from_yaml

In [ ]:
config_path = project_dir+"/configs/config.yaml"
cfg = get_config_from_yaml(config_path)
cfg

In [ ]:
cfg.visualizer.default_figsize

In [ ]:
from src.data import DataClass

In [ ]:
cfg.data.path = project_dir + "/" + cfg.data.path
cfg.data.index_col = "Date"
dataset = DataClass(cfg.data)

In [ ]:
dataset.train

In [ ]:
from src.visualizer import Visualizer

In [ ]:
help(Visualizer)

In [ ]:
cfg.visualizer.decomposition_model = 'multiplicative'

In [ ]:
vis = Visualizer(dataset, cfg.visualizer)

In [ ]:
vis.plot_autocorrelation("Beer Production (ML)")

In [ ]:
vis.plot_envelope_components("Beer Production (ML)")

In [ ]:
vis.plot_seasonal_decomposition("Beer Production (ML)")

In [ ]:
vis.plot_line_series("Beer Production (ML)")

In [ ]:
"""Orchestration script for executing and evaluating the stateless forecasting suite

on the Beer Production dataset within a Jupyter Notebook environment.
"""

import logging
import pandas as pd
import matplotlib.pyplot as plt

# Ensure logging outputs are visible in the notebook cell execution
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# Import the stateless models and configurations defined in src/models.py
# (Assuming the implementation from the previous step is saved in src/models.py)
from src.models import (
    DecompositionModel,
    ExponentialSmoothingModel,
    SARIMAModel,
    DecomposeConfig,
    ExponentialConfig,
    SARIMAConfig,
    ForecastingModelError
)


# =====================================================================
# 1. INITIALIZATION & CONFIGURATION SYNC
# =====================================================================
# Target configuration column from the dataset
TARGET_COL = "Beer Production (ML)"
FORECAST_STEPS = len(dataset.test)  # Forecast horizon: 8 quarters (2 years)

print(f"Index frequency detected in training data: {dataset.train.index.freqstr or dataset.train.index.inferred_freq}")
print(f"Total training observations: {len(dataset.train)}")

# Note on Frequency: The configuration loaded from YAML contains period=12 (Monthly).
# Since our dataset index is Quarterly, we can explicitly override the configuration 
# or pass it as is. Let's demonstrate building the execution payloads using the 
# configuration objects fetched from your central system 'cfg'.
try:
    # 1. Classical Decomposition Model Setup
    # Overriding period to None to let our base class automatically infer 4 (Quarterly)
    decompose_cfg = DecomposeConfig(model=cfg.models.decompose.model, period=None)
    decompose_model = DecompositionModel(model_config=decompose_cfg)

    # 2. Holt-Winters Exponential Smoothing Setup
    exponential_cfg = ExponentialConfig(
        trend=cfg.models.exponential_smoothing.trend,
        seasonal=cfg.models.exponential_smoothing.seasonal,
        seasonal_periods=None  # Set to None for automatic quarterly inference (4)
    )
    exponential_model = ExponentialSmoothingModel(model_config=exponential_cfg)

    # 3. Seasonal ARIMA (SARIMAX) Setup
    sarima_cfg = SARIMAConfig(
        p=cfg.models.sarima.p, d=cfg.models.sarima.d, q=cfg.models.sarima.q,
        P=cfg.models.sarima.P, D=cfg.models.sarima.D, Q=cfg.models.sarima.Q,
        s=None  # Set to None for automatic quarterly inference (4)
    )
    sarima_model = SARIMAModel(model_config=sarima_cfg)

    # =====================================================================
    # 2. EXECUTE FITTING ROUTINES (STATELESS WRAPPERS)
    # =====================================================================
    print("\n--- Fitting Models ---")
    decompose_model.fit(df=dataset.train, target_col=TARGET_COL)
    exponential_model.fit(df=dataset.train, target_col=TARGET_COL)
    sarima_model.fit(df=dataset.train, target_col=TARGET_COL)
    print("All models fitted successfully.")

    # =====================================================================
    # 3. GENERATE OUT-OF-SAMPLE FORECASTS
    # =====================================================================
    print(f"\n--- Generating Forecasts (Horizon = {FORECAST_STEPS} Steps) ---")
    forecast_decomp = decompose_model.predict(steps=FORECAST_STEPS)
    forecast_exp    = exponential_model.predict(steps=FORECAST_STEPS)
    forecast_sarima = sarima_model.predict(steps=FORECAST_STEPS)

    # Consolidate predictions into a single structured DataFrame for comparison
    forecast_df = pd.DataFrame({
        "Classical_Decomposition": forecast_decomp,
        "Holt_Winters_Exponential": forecast_exp,
        "Seasonal_ARIMA": forecast_sarima
    })
    
    print("\nConsolidated Out-of-Sample Predictions:")
    print(forecast_df)

    # =====================================================================
    # 4. VISUALIZATION INTEGRATION
    # =====================================================================
    # Using your visualizer class to inspect individual model outputs against actuals
    print("\n--- Plotting Results via Visualizer ---")
    
    # Example: Plotting Seasonal ARIMA forecasts
    vis.plot_predictions_vs_actuals(predictions=forecast_sarima, target_col=TARGET_COL)
    
    # Alternative: Generate a quick multi-model overlay comparative chart locally
    plt.figure(figsize=cfg.visualizer.default_figsize, dpi=cfg.visualizer.dpi)
    
    # Plot historical tail for context visual readability
    plt.plot(dataset.train[TARGET_COL].tail(24), label="Historical Actuals (Tail)", color="black", linewidth=2)
    
    # Overlay model predictions
    plt.plot(dataset.test[TARGET_COL], label="Actual Signal", linestyle="-")
    plt.plot(forecast_df["Classical_Decomposition"], label="Decomposition Forecast", linestyle="--")
    plt.plot(forecast_df["Holt_Winters_Exponential"], label="Holt-Winters Forecast", linestyle="-.")
    plt.plot(forecast_df["Seasonal_ARIMA"], label="SARIMA Forecast", linestyle=":")
    
    plt.title("Comparative Forecast Analysis - Beer Production (ML)", fontsize=14, fontweight='bold')
    plt.xlabel("Timeline Date", fontsize=12)
    plt.ylabel("Production Volume (ML)", fontsize=12)
    plt.legend(loc="upper left")
    plt.grid(True, linestyle="--", alpha=0.5)
    
    # Save the custom plot to the configured directory path
    plot_output_path = f"{cfg.visualizer.plot_path}model_comparison_overlay.png"
    plt.savefig(plot_output_path, bbox_inches="tight")
    plt.show()
    print(f"Comparison plot saved successfully to: {plot_output_path}")

except ForecastingModelError as fme:
    print(f"Forecasting Engine Execution Failure: {fme}")
except Exception as e:
    print(f"Unexpected Pipeline Failure: {e}")

In [ ]:
from src.metrics import ScoringEngine

In [ ]:
scorer = ScoringEngine(cfg.scoring)

In [ ]:
decomp_score = scorer.evaluate(dataset.test[TARGET_COL], forecast_decomp)
exp_score = scorer.evaluate(dataset.test[TARGET_COL], forecast_exp)
sarima_score = scorer.evaluate(dataset.test[TARGET_COL], forecast_sarima)

In [ ]:
pd.DataFrame(
    {
        "Decompositoin": decomp_score,
        "Holt Winter": exp_score,
        "SARIMA": sarima_score
    }
)